In [2]:
import os
from dotenv import load_dotenv
from llama_parse import LlamaParse

# .env dosyasını oku
load_dotenv()

pdf_path = "../data/raw/apple_10k.pdf"

# LlamaParse'ı başlatıyoruz
# result_type="markdown" parametresi tabloların formatını koruması için çok kritik
parser = LlamaParse(
    api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
    result_type="markdown", 
    verbose=True
)

print("PDF analiz ediliyor, tablolar Markdown'a çevriliyor. Bu işlem 1-2 dakika sürebilir...")
documents = parser.load_data(pdf_path)

print(f"✅ Döküman okundu! Toplam parça sayısı: {len(documents)}")

# İçeriğin tablo içeren bir kısmına göz atalım (Örn: ilk 1500 karakter)
print("\n--- Çıktı Önizlemesi ---")
print(documents[0].text[:1500])

/var/folders/mk/jj0k081x35zcyxnhns8rfzq00000gn/T/ipykernel_33469/3385409937.py:3: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


PDF analiz ediliyor, tablolar Markdown'a çevriliyor. Bu işlem 1-2 dakika sürebilir...
Started parsing the file under job_id 06a6bf4c-8546-4bb5-b7b8-dbb058bf321f
✅ Döküman okundu! Toplam parça sayısı: 80

--- Çıktı Önizlemesi ---

# UNITED STATES SECURITIES AND EXCHANGE COMMISSION

# Washington, D.C. 20549

# FORM 10-K

(Mark One)

☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the fiscal year ended September 24, 2022

or

☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from   to  .

Commission File Number: 001-36743

# Apple Inc.

(Exact name of Registrant as specified in its charter)

| California                        | 94-2404110                           |
| --------------------------------- | ------------------------------------ |
| (State or other jurisdiction      | (I.R.S. Employer Identification No.) |
| of incorporation or organization) |                        

In [3]:
from langchain_text_splitters import MarkdownTextSplitter
from langchain_core.documents import Document

# LlamaParse çıktısını LangChain'in anlayacağı tek bir uzun metne çeviriyoruz
full_text = "\n\n".join([doc.text for doc in documents])

# MarkdownTextSplitter tabloları ve başlıkları bozmamaya özen gösterir
# chunk_size: Her bir parçanın maksimum karakter boyutu
# chunk_overlap: Parçalar arasında bağlam kopmaması için kesişen karakter sayısı
splitter = MarkdownTextSplitter(chunk_size=2000, chunk_overlap=200)

# Metni parçalara (chunk'lara) ayırıyoruz
chunks = splitter.create_documents([full_text])

print(f"✅ Döküman toplam {len(chunks)} parçaya (chunk) bölündü.")
print("\n--- İlk Chunk Örneği ---")
print(chunks[0].page_content[:500])

✅ Döküman toplam 221 parçaya (chunk) bölündü.

--- İlk Chunk Örneği ---
# UNITED STATES SECURITIES AND EXCHANGE COMMISSION

# Washington, D.C. 20549

# FORM 10-K

(Mark One)

☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the fiscal year ended September 24, 2022

or

☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from   to  .

Commission File Number: 001-36743

# Apple Inc.

(Exact name of Registrant as specified in its charter)

| California          


In [4]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# 1. Metinleri vektöre çevirecek modeli tanımlıyoruz
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Qdrant bağlantısını kuruyoruz
qdrant_url = os.getenv("QDRANT_URL", "http://localhost:6333")
client = QdrantClient(url=qdrant_url)

collection_name = "apple_10k_reports"

# 3. Eğer veritabanında bu isimde bir tablo (koleksiyon) yoksa, yenisini oluşturuyoruz
# 1536, OpenAI'ın embedding modelinin boyutudur.
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    )

# 4. Parçaları vektörlere çevirip Qdrant'a kaydediyoruz
print("Vektörler oluşturuluyor ve Qdrant'a kaydediliyor. Bu işlem 15-20 saniye sürebilir...")
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

vector_store.add_documents(chunks)
print("✅ Bütün veriler Qdrant'a başarıyla indekslendi!")

Vektörler oluşturuluyor ve Qdrant'a kaydediliyor. Bu işlem 15-20 saniye sürebilir...
✅ Bütün veriler Qdrant'a başarıyla indekslendi!


In [5]:
# Veritabanından sorumuza en uygun (benzer) parçaları getirmesini istiyoruz
query = "What is the common stock par value for Apple?"
print(f"Sorulan Soru: {query}\n")

# En alakalı 2 parçayı (k=2) getir
results = vector_store.similarity_search(query, k=2)

for i, res in enumerate(results):
    print(f"--- En Alakalı Sonuç {i+1} ---")
    print(res.page_content[:400] + "...\n")

Sorulan Soru: What is the common stock par value for Apple?

--- En Alakalı Sonuç 1 ---
The Company’s common stock is traded on The Nasdaq Stock Market LLC under the symbol AAPL.

# Holders

As of October 14, 2022, there were 23,838 shareholders of record.

# Purchases of Equity Securities by the Issuer and Affiliated Purchasers

Share repurchase activity during the three months ended September 24, 2022 was as follows (in millions, except number of shares, which are reflected in thou...

--- En Alakalı Sonuç 2 ---
See accompanying Notes to Consolidated Financial Statements.

Apple Inc. | 2022 Form 10-K | 30



Apple Inc.
# CONSOLIDATED BALANCE SHEETS

(In millions, except number of shares which are reflected in thousands and par value)...



In [8]:
import sys
import os

# Notebook'un bulunduğu yerin bir üst klasörünü (proje ana dizinini) yola ekliyoruz
sys.path.append(os.path.abspath('..'))

# Şimdi yazdığımız modüller projeye dahil edilebilir
from src.ingestion.pdf_parser import parse_pdf_to_markdown
from src.ingestion.chunker import chunk_markdown_documents
from src.ingestion.indexer import index_documents, retrieve_documents

pdf_path = "../data/raw/apple_10k.pdf"
collection_name = "apple_10k_reports"

# 1. Pipeline'ı (Veri Boru Hattını) Uçtan Uca Çalıştırma
print("--- VERİ İŞLEME PIPELINE BAŞLIYOR ---")
docs = parse_pdf_to_markdown(pdf_path)
chunks = chunk_markdown_documents(docs)
index_documents(chunks, collection_name)

# 2. Test Etme
print("\n--- ARAMA TESTİ ---")
soru = "What is the common stock par value for Apple?"
cevaplar = retrieve_documents(soru, collection_name, k=1)

print(f"Soru: {soru}")
print(f"Gelen Kaynak Metin: \n{cevaplar[0].page_content[:300]}...")

--- VERİ İŞLEME PIPELINE BAŞLIYOR ---
[../data/raw/apple_10k.pdf] ayrıştırılıyor, lütfen bekleyin...
✅ Ayrıştırma tamamlandı. Toplam sayfa/bölüm: 80
✅ Metin başarıyla 221 parçaya bölündü.
Veriler vektörleştirilip Qdrant'a kaydediliyor...
✅ Veriler başarıyla indekslendi!

--- ARAMA TESTİ ---
Soru: What is the common stock par value for Apple?
Gelen Kaynak Metin: 
The Company’s common stock is traded on The Nasdaq Stock Market LLC under the symbol AAPL.

# Holders

As of October 14, 2022, there were 23,838 shareholders of record.

# Purchases of Equity Securities by the Issuer and Affiliated Purchasers

Share repurchase activity during the three months ended ...
